# 03 · 테스트 배치 비교 (Test05/06/07 한 번에)

여러 불량 테스트 파일을 **같은 모델**로 진단하고 F-score 등을 **비교표**로 출력한다.
학습에서 저장한 pca/scaler(정합 모드)를 재사용하므로 좌표계가 일치한다.

**주의**: 테스트 파일의 chg/dchg는 **학습 모델의 종류와 일치**해야 한다
(chg 모델 → chg 불량, dchg 모델 → dchg 불량).

In [ ]:
from function_def import *
from parameter import make_config
import os, math, numpy as np, pandas as pd, joblib
import tensorflow as tf
from pandas.plotting import register_matplotlib_converters
import matplotlib.pyplot as plt

## 설정 — 어떤 학습 모델로, 어떤 테스트 파일들을 볼지

`TRAIN_TAG`는 02_train에서 만든 실험 폴더명(예: 'chg_3files' = chg 정상 3개 합침).
`TESTS`에 (테스트CSV, 라벨CSV, 제목)들을 넣으면 순서대로 진단해 비교표를 만든다.

In [ ]:
SIGNAL_TYPE = 'chg'                 # 테스트 불량의 종류 (학습 모델과 일치!)
TRAIN_TAG   = 'chg_3files'          # 사용할 학습 모델 폴더 (checkpoints/<TRAIN_TAG>)
TEST_DIR    = './data/preprocessed/test'

# 비교할 테스트 파일들 (chg 예시). dchg면 파일명을 dchg로 바꿀 것.
TEST_CELLS = ['Test05_NG', 'Test06_NG', 'Test07_NG']
TESTS = []
for name in TEST_CELLS:
    csv   = os.path.join(TEST_DIR, '%s_%s.csv' % (name, SIGNAL_TYPE))
    label = os.path.join(TEST_DIR, '%s_%s_Label.csv' % (name, SIGNAL_TYPE))
    TESTS.append((csv, label, '%s_%s' % (name, SIGNAL_TYPE)))
for c, l, t in TESTS:
    print(t, "|", "csv 있음" if os.path.isfile(c) else "csv 없음!",
          "| label 있음" if os.path.isfile(l) else "| label 없음!")

## config 로드 + 학습 모델/PCA 경로 지정

In [ ]:
CFG = make_config(pca_mode='consistent')   # 학습과 동일 설정
ckpt_dir = os.path.join('checkpoints', TRAIN_TAG)
CFG['ckpt_dir']    = ckpt_dir
CFG['pca_path']    = os.path.join(ckpt_dir, 'pca.joblib')
CFG['scaler_path'] = os.path.join(ckpt_dir, 'scaler.joblib')

win_size, features_dim, feat_dim = CFG['win_size'], CFG['features_dim'], CFG['feat_dim']
latent_dim, batch_size, k_size   = CFG['latent_dim'], CFG['batch_size'], CFG['k_size']
lstm_units, drop_gen             = CFG['lstm_units'], CFG['dropout_rate_gen']
crit_filters, crit_drop          = CFG['critic_filters'], CFG['critic_dropout']
diffs_n, lags_n, smooth_n        = CFG['diffs_n'], CFG['lags_n'], CFG['smooth_n']
shape                   = CFG['shape']
encoder_input_shape     = CFG['encoder_input_shape']
encoder_reshape_shape   = CFG['encoder_reshape_shape']
generator_input_shape   = CFG['generator_input_shape']
generator_reshape_shape = CFG['generator_reshape_shape']
critic_x_input_shape    = CFG['critic_x_input_shape']
critic_z_input_shape    = CFG['critic_z_input_shape']
print("사용 모델:", ckpt_dir)

## 모델 1회 로드 (테스트마다 재사용)

가중치는 한 번만 불러오고, 여러 테스트 파일에 반복 적용한다.

In [ ]:
encoder   = build_encoder_layer(encoder_input_shape, encoder_reshape_shape,
                                win_size=win_size, latent_dim=latent_dim)
generator = build_generator_layer(generator_input_shape, generator_reshape_shape,
                                  win_size=win_size, features_dim=features_dim,
                                  lstm_units=lstm_units, dropout_rate=drop_gen)
critic_x  = build_critic_x_layer(critic_x_input_shape, k_size=k_size,
                                 filters=crit_filters, dropout_rate=crit_drop)
critic_z  = build_critic_z_layer(critic_z_input_shape)

z = Input(shape=(latent_dim, 1)); x = Input(shape=shape)
x_ = generator(z); z_ = encoder(x)
critic_x_model = Model([x, z], [critic_x(x), critic_x(x_), RandomWeightedAverage(batch_size)([x, x_])])
critic_z_model = Model([x, z], [critic_z(z), critic_z(z_), RandomWeightedAverage(batch_size)([z, z_])])
z_gen = Input(shape=(latent_dim, 1)); x_gen = Input(shape=shape)
x_gen_ = generator(z_gen); z_gen_ = encoder(x_gen); x_gen_rec = generator(z_gen_)
encoder_generator_model = Model([x_gen, z_gen], [critic_x(x_gen_), critic_z(z_gen_), x_gen_rec])

for nm, m in [('critic_x_model', critic_x_model), ('critic_z_model', critic_z_model),
              ('encoder_generator_model', encoder_generator_model)]:
    m.load_weights(os.path.join(ckpt_dir, nm + '.h5'))
pca_saved = joblib.load(CFG['pca_path'])
scaler_saved = joblib.load(CFG['scaler_path'])
print("model + pca/scaler loaded")

## 진단 함수 (한 파일 처리)

전처리 → PCA/scaler transform(학습 좌표계) → 예측 → 이상 구간 → 지표 계산.
결과 dict와 (Z_score, 구간)을 반환해 표와 그림에 모두 쓴다.

In [ ]:
def run_one(csv_path, label_path, title):
    df0 = pd.read_csv(csv_path)
    d1 = diff_smooth_df(df0, lags_n, diffs_n, smooth_n)
    data = pca_saved.transform(d1)              # 학습 PCA 재사용
    rows = [[i + 1] + [data[i][j] for j in range(features_dim)] for i in range(len(data))]
    df = pd.DataFrame(rows)
    df.columns = ['date'] + ['pca_%s' % str(i) for i in range(1, features_dim + 1)]

    X, index = time_segments_aggregate(df, interval=1, time_column='date')
    X = SimpleImputer().fit_transform(X)
    X = scaler_saved.transform(X)               # 학습 스케일러 재사용
    X, y, X_index, y_index = rolling_window_sequences(
        X, index, window_size=win_size, target_size=1, step_size=1, target_column=0)

    y_hat, critic = predict(X, encoder, generator, critic_x, shape, feat_dim)
    anomaly = Anomaly()
    final_scores, true_index, true, predictions = anomaly.score_anomalies(
        X, y_hat, critic, X_index, comb="mult")
    final_scores = np.array(final_scores)
    anomalies = anomaly.find_anomalies(final_scores, true_index)

    pred_length = len(final_scores)
    avg = np.average(final_scores)
    sigma = math.sqrt(np.sum((final_scores - avg) ** 2) / len(final_scores))
    Z = (final_scores - avg) / sigma
    pred_bin = [0] * pred_length
    for a in anomalies:
        s, e = int(a[0]), int(a[1])
        for k in range(s - 1, e):
            if 0 <= k < pred_length:
                pred_bin[k] = 1
    gt = np.array(pd.read_csv(label_path)['label'][:pred_length])
    pred = np.array(pred_bin)
    tp = int(np.sum((pred == 1) & (gt == 1))); tn = int(np.sum((pred == 0) & (gt == 0)))
    fp = int(np.sum((pred == 1) & (gt == 0))); fn = int(np.sum((pred == 0) & (gt == 1)))
    P = tp / (tp + fp) if tp + fp else 0
    R = tp / (tp + fn) if tp + fn else 0
    F = 2 * P * R / (P + R) if P + R else 0
    A = (tp + tn) / len(pred)
    res = {'test': title, 'Acc': A, 'Prec': P, 'Rec': R, 'F': F,
           'TP': tp, 'FP': fp, 'FN': fn, 'TN': tn, 'anom_ratio': float(np.mean(gt))}
    return res, Z, X, gt, pred

## 배치 실행 → 비교표

In [ ]:
results = []
plots = {}
for csv, label, title in TESTS:
    if not (os.path.isfile(csv) and os.path.isfile(label)):
        print("skip (파일 없음):", title); continue
    res, Z, X, gt, pred = run_one(csv, label, title)
    results.append(res)
    plots[title] = (Z, X, gt, pred)
    print("done:", title, "-> F=%.4f Recall=%.4f" % (res['F'], res['Rec']))

summary = pd.DataFrame(results)[['test', 'Acc', 'Prec', 'Rec', 'F', 'TP', 'FP', 'FN', 'TN', 'anom_ratio']]
summary = summary.round(4)
print("\n===== 비교표 (모델:", TRAIN_TAG, ") =====")
print(summary.to_string(index=False))

## (선택) 개별 시각화

특정 테스트의 Z-score/구간 그림을 보고 싶을 때 title만 바꿔 실행.

In [ ]:
title = TESTS[0][2]   # 예: 첫 번째 테스트
Z, X, gt, pred = plots[title]

def to_spans(b):
    sp, on, bg = [], False, 0
    for k, v in enumerate(b):
        if v == 1 and not on: on, bg = True, k
        elif v == 0 and on: sp.append((bg, k - 1)); on = False
    if on: sp.append((bg, len(b) - 1))
    return sp

register_matplotlib_converters()
L = len(pred); ml = L - 10; t = range(ml)
Xs = np.array([X[k, 1] for k in range(ml)])
plt.figure(figsize=(30, 12))
plt.plot(t, 3 * Xs[:, 0], label='3*PCA1')
plt.plot(t, 3 * Xs[:, 1], label='3*PCA2')
plt.plot(t, Z[:ml], label='Z score')
plt.legend(loc=0, fontsize=30)
for i, span in enumerate([to_spans(gt), to_spans(pred)]):
    for a in span:
        plt.axvspan(a[0], a[1], color=['red', 'blue'][i], alpha=0.2)
plt.title(' {} : Red=True, Blue=Pred'.format(title), size=34)
plt.ylabel('PCA1, PCA2, Z_score', size=30); plt.xlabel('Time', size=30)
plt.xticks(size=26); plt.yticks(size=26); plt.xlim([t[0], t[-1]]); plt.show()